In [35]:
import pandas as pd
import numpy as np 
import sys
import os
from pathlib import Path 
from tqdm import tqdm 

In [70]:
path_to_results = Path("../../project_folder/results/mi_experiments/classifier/gaussians")

In [71]:
results_baseline_clf = {"mse_test": [],
                        "mse_val": [],
                        "num_dims": [], 
                        "hidden_dim": [],
                        "hidden_layers": [], 
                        "iteration": [],
                        "loc": []
                       }

for folder in tqdm(os.listdir(path_to_results)):
    folder_split = folder.split("_")
    loc, dimensions, hidden_dim, hidden_layers_no, iteration = (float(folder_split[1]), 
                                                                int(folder_split[3]), 
                                                                int(folder_split[6]), 
                                                                int(folder_split[10]),
                                                                int(folder_split[12]))
    metrics = pd.read_csv(path_to_results / folder / "metrics.csv")
    val_mse = metrics.mse_log_ratios_val[0].item()
    test_mse = metrics.mse_log_ratios_test[0].item()
    results_baseline_clf["mse_test"].append(test_mse)
    results_baseline_clf["mse_val"].append(val_mse)
    results_baseline_clf["num_dims"].append(dimensions)
    results_baseline_clf["hidden_dim"].append(hidden_dim)
    results_baseline_clf["hidden_layers"].append(hidden_layers_no)
    results_baseline_clf["iteration"].append(iteration)
    results_baseline_clf["loc"].append(loc)

100%|██████████| 360/360 [00:06<00:00, 58.14it/s]


In [72]:
results_baseline_clf_df = pd.DataFrame(results_baseline_clf)

In [73]:
# Average out the iteration axis 
df_mean = (
    results_baseline_clf_df
    .groupby(["hidden_dim", "hidden_layers", "num_dims", "loc"], as_index=False)
    .mean()   
)

In [74]:
best_configs = df_mean.loc[
    df_mean.groupby(["loc", "num_dims"])["mse_val"].idxmin()
]

In [75]:
best_configs

,hidden_dim,hidden_layers,num_dims,loc,mse_test,mse_val,iteration
40,128,0,2,1.0,0.000173,0.000173,1.0
42,128,0,5,1.0,0.001535,0.001520,1.0
44,128,0,10,1.0,0.009180,0.009033,1.0
46,128,0,20,1.0,0.071503,0.073151,1.0
48,128,0,30,1.0,1.009372,1.000282,1.0
1,64,0,2,2.0,0.000278,0.000278,1.0
83,256,0,5,2.0,0.012290,0.012031,1.0
45,128,0,10,2.0,16.580688,16.738263,1.0
37,64,3,20,2.0,145.286225,144.979234,1.0
119,256,3,30,2.0,544.635217,541.797891,1.0


In [76]:
best_configs_to_save = best_configs.copy()
best_configs_to_save["value"] = best_configs["mse_test"]
best_configs_to_save = best_configs_to_save.drop(columns=["mse_test", "mse_val", "iteration"])

In [78]:
best_configs_to_save.to_csv(path_to_results / "gaussian_results.csv")